## Import libraries


In [17]:
import os
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchcrop
from torchcrop import (
    CropParameters,
    SoilParameters,
    SiteParameters,
    Lintul5Model,
    WeatherDriver
)

DATA_DIR = Path("data", "brandenburg", "torchcrop")

## Prepare dataset and dataloader


In [ ]:
class TorchCropDataset(Dataset):
    """Reads per-location weather, soil, and site data for torchcrop.

    Directory layout (under ``DATA_DIR``)::

        weather/<location>.csv   daily forcing, one file per location
        soil/soil.csv            one row per location, indexed by ``location``
        site/site.csv            one row per location, indexed by ``location``

    ``__getitem__`` returns a single sample as a dict with:

        * ``weather`` : ``[T, 8]`` float tensor in torchcrop channel order
          (DOY, davtmp, tmin, tmax, irrad, rain, vp, wind)
        * ``soil``    : dict of scalar floats keyed by ``SoilParameters`` field
        * ``site``    : dict of scalar floats keyed by ``SiteParameters`` field

    Use `collate_torchcrop` as the DataLoader ``collate_fn`` to assemble a
    batch into ready-to-run ``WeatherDriver`` / ``SoilParameters`` /
    ``SiteParameters`` objects.
    """

    # CSV weather columns, in the exact order torchcrop expects them.
    WEATHER_COLS = [
        "Date",          # -> doy
        "TempMean",      # -> davtmp
        "TempMin",       # -> tmin
        "TempMax",       # -> tmax
        "Radiation",     # -> irrad
        "Precipitation", # -> rain
        "VapPressure",   # -> vp
        "Windspeed",     # -> wind
    ]

    # soil.csv column -> SoilParameters field name
    SOIL_MAP = {
        "SMDRY": "wcad",      # air-dry water content
        "SMW": "wcwp",        # wilting point
        "SMFC": "wcfc",       # field capacity
        "SMO": "wcst",        # saturation
        "CRAIRC": "crairc",   # critical air content
        "SMI": "wci",         # initial root-zone moisture
        "SMLOWI": "wci_lower",
        "RDMSO": "rdmso",     # max rooting depth from soil [m]
        "RUNFR": "runfr",
        "CFEV": "cfev",
        "KSUB": "ksub",
        "NMINS": "nmini",
        "PMINS": "pmini",
        "KMINS": "kmini",
        "RTNMINS": "rtnmins",
        "RTPMINS": "rtpmins",
        "RTKMINS": "rtkmins",
    }

    # site.csv column -> SiteParameters field name
    SITE_MAP = {
        "LATITUDE": "latitude",
        "ALTITUDE": "altitude",
        "IDPL": "idpl",
        "CO2": "co2",
    }

    def __init__(self, weather_dir, soil_dir, site_dir, dtype=torch.float32):
        super().__init__()
        self.weather_dir = Path(weather_dir)
        self.dtype = dtype
        self.soil_data = pd.read_csv(
            os.path.join(soil_dir, "soil.csv")
        ).set_index("location")
        self.site_data = pd.read_csv(
            os.path.join(site_dir, "site.csv")
        ).set_index("location")
        # Locations present in both soil and site tables.
        self.locations = self.soil_data.index.intersection(self.site_data.index)

    def __len__(self):
        return len(self.locations)

    def _load_weather(self, location):
        df = pd.read_csv(
            self.weather_dir / f"{location}.csv", parse_dates=["Date"]
        )
        df = df[self.WEATHER_COLS].copy()
        df["Radiation"] = df["Radiation"] / 1000.0   # kJ -> MJ m-2 d-1
        df["Date"] = df["Date"].dt.dayofyear          # date -> day-of-year
        return torch.as_tensor(df.values, dtype=self.dtype)  # [T, 8]

    def __getitem__(self, idx):
        location = self.locations[idx]

        weather = self._load_weather(location)

        srow = self.soil_data.loc[location]
        soil = {field: float(srow[col]) for col, field in self.SOIL_MAP.items()}

        trow = self.site_data.loc[location]
        site = {field: float(trow[col]) for col, field in self.SITE_MAP.items()}

        return {"weather": weather, "soil": soil, "site": site}


def collate_torchcrop(batch, dtype=torch.float32):
    """Collate samples into batched torchcrop driver/parameter objects.

    Args:
        batch: list of samples produced by `TorchCropDataset.__getitem__`.
        dtype: target dtype for the assembled tensors.

    Returns:
        Tuple ``(weather, soil_params, site_params)`` where ``weather`` is a
        `WeatherDriver` of shape ``[B, T, 8]`` and ``soil_params`` /
        ``site_params`` are batched dataclasses with ``[B]`` scalar fields.
    """
    # Weather: [B, T, 8]
    weather = torch.stack([s["weather"] for s in batch], dim=0).to(dtype)
    weather = WeatherDriver(weather)

    # Soil / site: stack each field across the batch into a [B] tensor.
    soil_fields = batch[0]["soil"].keys()
    soil_kwargs = {
        f: torch.tensor([s["soil"][f] for s in batch], dtype=dtype)
        for f in soil_fields
    }
    soil_params = SoilParameters(**soil_kwargs)

    site_fields = batch[0]["site"].keys()
    site_kwargs = {
        f: torch.tensor([s["site"][f] for s in batch], dtype=dtype)
        for f in site_fields
    }
    site_params = SiteParameters(**site_kwargs)

    return weather, soil_params, site_params

In [ ]:
# Prepare the dataset and dataloader
dataset = TorchCropDataset(
    weather_dir=DATA_DIR / "weather",
    soil_dir=DATA_DIR / "soil",
    site_dir=DATA_DIR / "site",
)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_torchcrop,
)

print(f"Dataset: {len(dataset)} locations")

Dataset: 18 locations


In [14]:
weather.data.shape

torch.Size([4, 731, 8])

In [16]:
site_params

SiteParameters(latitude=tensor([53.2199, 53.0662, 53.0128, 52.8835]), altitude=tensor([68., 68., 68., 68.]), co2=tensor([370., 370., 370., 370.]), cfet=tensor(1.), fpenmtb=tensor([[4.0000e+01, 1.0500e+00],
        [3.6000e+02, 1.0000e+00],
        [7.2000e+02, 9.5000e-01],
        [1.0000e+03, 9.2000e-01],
        [2.0000e+03, 9.2000e-01]]), plant_at_sowing=tensor(1.), idpl=tensor([270., 270., 270., 270.]), idem=tensor(0.))

In [11]:
# Inspect one batch to confirm the data is read and assembled correctly.
weather, soil_params, site_params = next(iter(dataloader))

print("weather :", tuple(weather.data.shape), weather.data.dtype, "[B, T, C]")
print("doy[0, :5]      :", weather.channel("doy")[0, :5].tolist())
print("irrad[0, :5] MJ :", weather.channel("irrad")[0, :5].tolist())
print()
print("soil.wcfc  [B]:", soil_params.wcfc.tolist())
print("soil.wcst  [B]:", soil_params.wcst.tolist())
print("soil.rdmso [B]:", soil_params.rdmso.tolist())
print()
print("site.latitude [B]:", site_params.latitude.tolist())
print("site.co2      [B]:", site_params.co2.tolist())
print("site.idpl     [B]:", site_params.idpl.tolist())

weather : (4, 731, 8) torch.float32 [B, T, C]
doy[0, :5]      : [1.0, 2.0, 3.0, 4.0, 5.0]
irrad[0, :5] MJ : [3.114000082015991, 3.8989999294281006, 1.496000051498413, 1.7259999513626099, 1.656000018119812]

soil.wcfc  [B]: [0.13239166140556335, 0.148076593875885, 0.23943452537059784, 0.07972655445337296]
soil.wcst  [B]: [0.37557506561279297, 0.3917401432991028, 0.4169130325317383, 0.3723953664302826]
soil.rdmso [B]: [2.0, 2.0, 2.0, 2.0]

site.latitude [B]: [53.21989822387695, 53.066200256347656, 53.01279830932617, 52.88349914550781]
site.co2      [B]: [370.0, 370.0, 370.0, 370.0]
site.idpl     [B]: [270.0, 270.0, 270.0, 270.0]


## Spin-up and sowing

The weather series starts on **1 January** but the crop is sown on
**day-of-year 270** (`site.csv` → `IDPL`). torchcrop honours `idpl` through a
*sowing latch* (`ModelState.sown`): days 1–269 are a spin-up period during
which only the **soil water balance** evolves, while the emergence and
vernalisation thermal clocks stay frozen. The latch turns on once
`doy >= idpl` and never resets, so it survives the year-boundary wraparound
(autumn sowing → summer harvest).

Two requirements when running the model:

* Pass **`start_doy=1`** so the engine's day-of-year aligns with the weather
  (DOY is derived from `start_doy` + day index, *not* from the weather DOY
  column).
* Keep `idpl` in `site_params` — the `DataLoader` already sets it from
  `site.csv`.

```python
model = Lintul5Model(crop_params, soil_params, site_params)
output = model(weather, start_doy=1)   # sowing happens internally at doy 270
```

With the default `idpl=0`, sowing coincides with the first simulated day
(legacy behaviour), so existing single-season runs are unaffected.
